# 🎯 [QLT04_M07] 정비 자원 우선 투입 결정을 위한 군수참모처 HITL 정책 결정 모듈 설계 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m07_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M07 |
| **미션 주제** | 정비 자원 우선 투입 결정을 위한 군수참모처 HITL 정책 결정 모듈 설계 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 정비 자원 우선 투입 결정을 위한 군수참모처 HITL 정책 결정 모듈 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

정비 자원 우선 투입 결정을 위한 군수참모처 HITL 정책 결정 모듈을 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **정비 자원 우선 투입 안을 정비담당-정비관-품질검사관-군수참모 순서로 사람이 검토·결정하는 HITL 워크플로우를 설계하고 권한 통제와 이력 무결성을 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **역할별 권한(RBAC)과 단계 순서를 정의하고 시스템은 제안만, 확정은 사람만 하도록 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/workflow/qlt04_approval_system.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 보안 규정 및 행정 절차 부합성

#### 미션 PASS 조건

- 정량 기준: 이력 기록 무결성 100%, 승인 권한 우회 시도 차단율 100%


- 진행 기준: 결재선 승인/반려 시뮬레이션 및 이력 조회 화면 구현


- 검증 방법: 권한별 접근 제어(RBAC) 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""저가동 장비의 정비 자원 우선 투입안을 정비담당 상신 -> 정비관 검토 -> 품질검사관 검토 -> 군수참모 결재의 HITL 워크플로우로 처리하되, 시스템 자동 확정 금지·반려 사유 필수화·재기안 연계·권한 우회 차단(RBAC)과 해시 체인 감사 로그의 이력 무결성·위변조 탐지를 검증한다."""
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# ---- 정비·검사 병목 판별 룰 설정(YAML 형식 문자열: 기준이 바뀌면 코드가 아니라 이 설정만 고친다) ----
# clause는 룰과 대응시킨 '가상 품질지침 조항' 예시이며 실제 규정 원문이 아니다.
RULES_YAML = """
- {id: Q01, name: 부품 재고 가용률 50% 미만, field: parts_stock_pct, op: "<", value: 50, weight: 2, clause: 가상품질지침 제3조}
- {id: Q02, name: 정비병 충원율 60% 미만, field: mechanic_fill_pct, op: "<", value: 60, weight: 2, clause: 가상품질지침 제4조}
- {id: Q03, name: 부품 대기 8일 초과, field: parts_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제5조}
- {id: Q04, name: 정비병 부족 대기 8일 초과, field: labor_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제6조}
- {id: Q05, name: 외주 정비 15일 초과, field: outsource_days, op: ">", value: 15, weight: 2, clause: 가상품질지침 제7조}
- {id: Q06, name: 외주 검사 대기 6일 초과, field: inspect_wait_days, op: ">", value: 6, weight: 3, clause: 가상품질지침 제8조}
- {id: Q07, name: 재작업 1건 이상, field: rework_count, op: ">=", value: 1, weight: 3, clause: 가상품질지침 제9조}
- {id: Q08, name: 재작업률 25% 초과, field: rework_rate, op: ">", value: 0.25, weight: 2, clause: 가상품질지침 제9조}
- {id: Q09, name: 규격·제원 불일치 3건 이상, field: spec_mismatch, op: ">=", value: 3, weight: 3, clause: 가상품질지침 제10조}
- {id: Q10, name: 하자·불만 3건 이상, field: defect_reports, op: ">=", value: 3, weight: 2, clause: 가상품질지침 제11조}
- {id: Q11, name: 장비 연령 15년 초과, field: age_years, op: ">", value: 15, weight: 1, clause: 가상품질지침 제12조}
- {id: Q12, name: 정비 요청 6건 이상, field: maint_orders, op: ">=", value: 6, weight: 1, clause: 가상품질지침 제13조}
- {id: Q13, name: 대기 일수 합계 30일 초과, field: total_wait, op: ">", value: 30, weight: 3, clause: 가상품질지침 제14조}
- {id: Q14, name: 검사 대기가 외주 정비 일수의 80% 초과, field: insp_ratio, op: ">", value: 0.8, weight: 2, clause: 가상품질지침 제8조}
- {id: Q15, name: 규격 불일치 2건 이상과 하자 접수 동반, field: spec_defect, op: "==", value: true, weight: 3, clause: 가상품질지침 제10조}
- {id: Q16, name: 최근 90일 가동 시간 150시간 미만, field: op_hours, op: "<", value: 150, weight: 1, clause: 가상품질지침 제15조}
"""
GRADE_CUT = [(10, "고위험"), (6, "주의"), (3, "관심")]
OPS = {"<": lambda a, b: a < b, ">": lambda a, b: a > b, "<=": lambda a, b: a <= b, ">=": lambda a, b: a >= b, "==": lambda a, b: a == b}


def parse_rules(text):
    """YAML 한 줄 맵({key: value, ...}) 형식 룰 문자열을 읽는다(외부 yaml 라이브러리 없이)."""
    out = []
    for line in text.strip().splitlines():
        body = line.strip()[3:-1]
        r = {}
        for part in body.split(", "):
            k, v = part.split(": ", 1)
            v = v.strip().strip('"')
            r[k] = True if v == "true" else False if v == "false" else (float(v) if v.replace(".", "", 1).isdigit() else v)
        out.append(r)
    return out


def derive_rec(rec):
    """룰이 쓰는 파생 필드(대기 합계, 재작업률, 검사 대기 비율(외주 5일 이상일 때), 규격 불일치 2건 이상+하자 동반)를 추가한다."""
    r = dict(rec)
    r["total_wait"] = r["parts_wait_days"] + r["labor_wait_days"] + r["outsource_days"] + r["inspect_wait_days"]
    r["rework_rate"] = r["rework_count"] / r["maint_orders"] if r["maint_orders"] else 0.0
    r["insp_ratio"] = r["inspect_wait_days"] / r["outsource_days"] if r["outsource_days"] >= 5 else 0.0
    r["spec_defect"] = bool(r["spec_mismatch"] >= 2 and r["defect_reports"] >= 1)
    return r


def apply_rules(rec, rules, derived=False):
    """룰 적용: (점수, 적중 룰 id 목록). derived=True면 이미 파생 필드가 있는 레코드(경계 시험용)로 본다."""
    r = rec if derived else derive_rec(rec)
    hits = [x["id"] for x in rules if OPS[x["op"]](r[x["field"]], x["value"])]
    w = {x["id"]: x["weight"] for x in rules}
    return sum(w[h] for h in hits), hits


def grade_of(score):
    for cut, g in GRADE_CUT:
        if score >= cut:
            return g
    return "정상"

# ---- 가동률 회귀 모델과 저가동 요인 분류·추천 모델(외부 AI 서비스 없이 scikit-learn 사용) ----
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CAUSE_COLS = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch",
              "defect_reports", "parts_wait_days", "labor_wait_days"]
ACTION = {"부품 대기": "부품 사전 청구·재고 보강", "정비병 부족": "정비병 재배치·교육 일정 조정", "외주 정비": "외주 정비 일정 조율·내부 정비 확대", "검사·재작업": "검사 일정 조율·규격 사전 대조"}


def cause_frame(df):
    x = df[CAUSE_COLS].copy()
    x["out_insp"] = x["outsource_days"] + x["inspect_wait_days"] + 4 * x["rework_count"]
    x["wait_pm"] = (x["parts_wait_days"] + x["labor_wait_days"]) / (x["maint_orders"] + 1)
    return x


def fit_avail(df, alpha=1.0):
    """가동률(%) 회귀: 표준화 + 릿지 회귀."""
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(feat_frame(df), df["avail_pct"])


def fit_cause(df, cut=85, C=5):
    """가동률이 cut% 미만인 장비의 주요 저가동 요인(4종)을 분류한다."""
    low = df[(df["avail_pct"] < cut) & (df["main_cause"] != "없음")]
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(cause_frame(low), low["main_cause"])


SIM_FEAT = ["age_years", "maint_orders", "spec_mismatch", "stock_gap_x_orders", "fill_gap_x_orders", "parts_stock_pct", "mechanic_fill_pct", "etype_B", "etype_C"]


def sim_frame(df):
    """자원 재배치 시뮬레이션용 설명 변수: 부족분이 클수록 대기가 더 빠르게 늘고 정비 요청 건수와 곱해져 커진다는 구조를 제곱·상호작용 항으로 둔다."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    x["stock_gap_x_orders"] = x["maint_orders"] * ((100 - x["parts_stock_pct"]) / 28) ** 2
    x["fill_gap_x_orders"] = x["maint_orders"] * ((100 - x["mechanic_fill_pct"]) / 24) ** 2
    return x[SIM_FEAT]


def fit_sim(df, alpha=0.5):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(sim_frame(df), df["avail_pct"])

# ---- 역할과 권한(최소 권한 원칙): 시스템은 '제안'만 하고 확정 권한이 없다 ----
PERM = {"시스템": {"propose"}, "정비담당": {"submit", "redraft", "view"}, "정비관": {"review_maint", "reject", "view"}, "품질검사관": {"review_quality", "reject", "view"},
        "군수참모": {"approve", "reject", "view"}, "감사관": {"view", "audit"}, "부서장": {"submit", "approve", "reject", "view"}}      # 부서장은 상신·결재를 겸할 수 있으나 같은 건은 분리
NEXT = {"제안": ("submit", "상신"), "상신": ("review_maint", "정비관검토"), "정비관검토": ("review_quality", "품질검사"), "품질검사": ("approve", "승인")}
USERS = {"sys": "시스템", "kim": "정비담당", "lee": "정비관", "park": "품질검사관", "choi": "군수참모", "jung": "감사관", "oh": "부서장"}
MAX_REDRAFT = 2


class Chain:
    """해시 체인 감사 로그: 각 기록이 직전 기록의 해시를 포함해 중간 수정이 드러난다."""
    def __init__(self):
        self.log = []

    def add(self, **body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"seq": len(self.log), **body}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest()})

    def verify(self):
        prev = "GENESIS"
        for e in self.log:
            b = {k: v for k, v in e.items() if k not in ("prev", "hash")}
            if e["prev"] != prev or hashlib.sha256((prev + json.dumps(b, sort_keys=True, ensure_ascii=False)).encode()).hexdigest() != e["hash"]:
                return e["seq"]
            prev = e["hash"]
        return None


class Workflow:
    def __init__(self):
        self.audit, self.cases, self.blocked = Chain(), {}, []

    def _deny(self, who, case, action, why):
        self.audit.add(event="DENIED", who=who, case=case, action=action, reason=why)
        self.blocked.append((who, case, action, why)); return False, why

    def act(self, who, case, action, reason="", evidence=""):
        """
        [TODO : 학생 실습 7] 군수참모처 HITL 정책 결정 및 RBAC(역할기반 접근통제) 전이 로직을 완성하세요.
        - 힌트 1: USERS에서 사용자의 role을 조회하고, 없으면 self._deny(who, case, action, "미등록 사용자")를 반환합니다.
        - 힌트 2: action == "delete_log" 시도 시 감사 로그 삭제 금지 사유로 _deny 처리합니다.
        - 힌트 3: action not in PERM[role]인 경우 권한 없음으로 _deny 처리합니다.
        - 힌트 4: 상태 전이 규칙:
          * propose: 제안 등록 (version=1, state='제안')
          * reject: 반려 사유 및 근거 필수, state='종결(반려)'
          * approve: 결재 승인, 최종 state='승인'
        - 반환값: (성공여부: bool, 메시지: str)
        """
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.

wf = Workflow()
rows = []
for k, ((_, p), cause) in enumerate(zip(picked.iterrows(), rec_cause), 1):
    case = f"C{k:02d}"
    wf.act("sys", case, "propose", reason=f"{p['equip_id']} 룰 점수 {p['score']:.0f}, 추천 요인 {cause} -> {ACTION[cause]}")
    wf.act("kim", case, "submit", "정비 자원 우선 투입 검토 요청", f"룰 적중 {','.join(p['hits'][:4])}, 추천 조치 {ACTION[cause]}")
    q_issue = sorted(QUALITY & set(p["hits"])) if len(QUALITY & set(p["hits"])) >= 2 else []                              # 품질검사관이 볼 문제
    m_issue = (p["avail_pct"] < 40 and cause == "정비병 부족")                # 정비관이 볼 문제: 인력 이동 시 제공 부대 영향 소명 필요
    note = "군수참모 결정"
    for _ in range(14):
        state = wf.cases[case]["state"]
        if state in ("승인", "종결(반려)"):
            break
        if state == "상신":
            if m_issue:
                wf.act("lee", case, "reject", "인력 이동 안은 제공 부대 영향 소명 필요", f"가동률 {p['avail_pct']}%, 요인 {cause}")
            else:
                wf.act("lee", case, "review_maint", "정비 능력·일정 검토 완료", f"추천 조치 {ACTION[cause]} 수행 가능")
        elif state == "정비관검토":
            if rng.random() < 0.15:
                note = "자동 확정 없음: 품질검사 응답 대기"; break                  # 무응답은 보류로 둔다
            if q_issue:
                wf.act("park", case, "reject", f"규격·검사 조건 보완 필요({','.join(q_issue)})", "재작업·규격 불일치·검사 대기 항목 점검")
            else:
                wf.act("park", case, "review_quality", "품질검사 검토 완료", "규격·검사 항목 이상 없음 또는 보완 확인")
        elif state == "품질검사":
            wf.act("choi", case, "approve", "승인: 자원 우선 투입 진행", f"룰 점수 {p['score']:.0f}, 검토 단계 완료")
        elif state == "반려":
            wf.act("kim", case, "redraft", "반려 사유 반영해 수정", f"보완 대상 {q_issue or ['제공 부대 영향 소명']}, 재기안 {wf.cases[case]['version']}차")
            m_issue = False                                                 # 제공 부대 영향 소명서를 첨부(모의)
            if rng.random() < 0.65:                                          # 재기안 후 보완이 끝날 확률(모의)
                q_issue = []
    final = wf.cases[case]["state"]
    if final not in ("승인", "종결(반려)") and note.startswith("자동"):
        final = "보류(무응답)"
    elif final == "종결(반려)":
        note = "재기안 한도(2회) 초과"
    rows.append((case, p["equip_id"], final, wf.cases[case]["version"], note))
print("[결재 결과] 12건 (모의 결재자 기준, 실제 결재는 사람이 한다)")
print(pd.DataFrame(rows, columns=["건", "장비", "최종 상태", "버전", "비고"]).to_string(index=False))
st = pd.Series([r[2] for r in rows]).value_counts().to_dict()
print("상태 집계:", st, f"/ 재기안 {sum(e['event'] == 'REDRAFT' for e in wf.audit.log)}회, 반려 이력 {sum(e['event'] == 'REJECTED' for e in wf.audit.log)}건(모두 사유·근거 기록)")

# ---- 권한 우회·절차 위반 시도 ----
wf.act("sys", "T01", "propose", reason="시험 건"); wf.act("kim", "T01", "submit", "시험 상신", "근거")
attempts = [
    ("시스템 자동 승인", lambda: wf.act("sys", "T01", "approve", "자동", "자동")),
    ("정비담당이 정비관 검토 대행", lambda: wf.act("kim", "T01", "review_maint", "대행", "근거")),
    ("검토 단계 건너뛰고 승인", lambda: wf.act("choi", "T01", "approve", "바로 승인", "근거")),
    ("미등록 사용자 접근", lambda: wf.act("guest", "T01", "review_maint", "열람", "근거")),
    ("사유 없는 검토", lambda: wf.act("lee", "T01", "review_maint", "", "")),
    ("사유 없는 반려", lambda: wf.act("lee", "T01", "reject", "", "근거")),
    ("감사 로그 삭제", lambda: wf.act("kim", "T01", "delete_log", "정리", "근거")),
    ("감사관이 승인 시도", lambda: wf.act("jung", "T01", "approve", "승인", "근거")),
    ("종결된 건 재승인", lambda: wf.act("choi", next(c for c, v in wf.cases.items() if v["state"] == "승인"), "approve", "재승인", "근거")),
    ("존재하지 않는 건 승인", lambda: wf.act("choi", "X99", "approve", "승인", "근거")),
]
wf.act("sys", "T02", "propose", reason="시험"); wf.act("oh", "T02", "submit", "상신", "근거")
wf.act("lee", "T02", "review_maint", "정비 검토", "이상 없음"); wf.act("park", "T02", "review_quality", "품질 검토", "이상 없음")
attempts.append(("상신자가 본인 건 승인(분리 원칙)", lambda: wf.act("oh", "T02", "approve", "본인 승인", "근거")))
wf.act("sys", "T03", "propose", reason="시험"); wf.act("kim", "T03", "submit", "상신", "근거"); wf.act("lee", "T03", "reject", "수정 필요", "근거")
attempts.append(("반려 건을 재기안 없이 검토 진행", lambda: wf.act("lee", "T03", "review_maint", "그대로 진행", "근거")))
attempts.append(("타인이 대신 재기안", lambda: wf.act("oh", "T03", "redraft", "대신 수정", "근거")))
print("\n[권한 우회·절차 위반 시도]")
blocked = 0
for name, fn in attempts:
    ok, why = fn(); blocked += (not ok)
    print(f"  {name:22s}-> {'차단' if not ok else '통과(결함)'}: {why}")
print(f"우회·위반 시도 차단 {blocked}/{len(attempts)} ({blocked / len(attempts):.0%}) (목표 100%: {'달성' if blocked == len(attempts) else '미달'})")

# ---- RBAC 매트릭스와 이력 무결성 ----
acts = ["propose", "submit", "review_maint", "review_quality", "approve", "reject", "redraft", "audit"]
mx = pd.DataFrame([[("O" if a in PERM[r] else "-") for a in acts] for r in PERM], index=list(PERM), columns=["제안", "상신", "정비검토", "품질검토", "승인", "반려", "재기안", "감사"])
print("\n[역할별 권한 매트릭스(O 허용 / - 불가)]"); print(mx.to_string())
bad = wf.audit.verify()
print(f"\n감사 로그 {len(wf.audit.log)}건, 해시 체인 검증: {'정상' if bad is None else f'seq {bad} 불일치'} -> 이력 무결성 {'100%' if bad is None else '훼손'}(목표 100%: {'달성' if bad is None else '미달'})")
idx = 10; saved = json.dumps(wf.audit.log[idx], ensure_ascii=False)
wf.audit.log[idx]["reason"] = "사후 수정"; v = wf.audit.verify()
print(f"{idx}번째 기록 임의 수정 후 검증: {'위변조 탐지(seq ' + str(v) + ')' if v is not None else '탐지 실패'}")
wf.audit.log[idx] = json.loads(saved)
print("\n[이력 조회 화면] C02 건")
for e in wf.audit.log:
    if e.get("case") == "C02":
        print(f"  #{e['seq']:03d} {e['event']:14s} {e.get('role', e['who']):6s} v{e.get('version', 1)} | {e.get('reason', '')[:34]}")
print("※ 정비 우선순위와 자원 투입 결정은 사람(정비관·품질검사관·군수참모)이 하며, 시스템은 위험 표시와 추천만 제공한다. 결재선은 소속 기관 규정을 따른다.")

plt.figure(figsize=(6, 3.5)); EN = {"승인": "approved", "반려": "rejected", "종결(반려)": "closed-rejected", "보류(무응답)": "on hold"}
plt.bar([EN.get(k, k) for k in st], list(st.values()), color="tab:blue"); plt.title("HITL workflow outcomes (12 cases)"); plt.ylabel("cases"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[결재 결과] 12건 (모의 결재자 기준, 실제 결재는 사람이 한다)
  건     장비   최종 상태  버전                   비고
C01 E-0047 보류(무응답)   2 자동 확정 없음: 품질검사 응답 대기
C02 E-0231      승인   2              군수참모 결정
C03 E-0041      승인   2              군수참모 결정
C04 E-0050 보류(무응답)   1 자동 확정 없음: 품질검사 응답 대기
C05 E-0220      승인   2              군수참모 결정
C06 E-0230      승인   1              군수참모 결정
C07 E-0095      승인   1              군수참모 결정
C08 E-0109      승인   2              군수참모 결정
C09 E-0163      승인   2              군수참모 결정
C10 E-0076      승인   2              군수참모 결정
C11 E-0028      승인   3              군수참모 결정
C12 E-0233      승인   2              군수참모 결정
상태 집계: {'승인': 10, '보류(무응답)': 2} / 재기안 10회, 반려 이력 10건(모두 사유·근거 기록)

[권한 우회·절차 위반 시도]
  시스템 자동 승인             -> 차단: 권한 없음(시스템 역할에 approve 권한 없음)
  정비담당이 정비관 검토 대행       -> 차단: 권한 없음(정비담당 역할에 review_maint 권한 없음)
  검토 단계 건너뛰고 승인         -> 차단: 단계 위반(현재 상신, 필요 행위 review_maint)
  미등록 사용자 접근            -> 차단: 미등록 사용자
  사유 없는 검토              -> 차단: 사유·근거 입력 필수(반려 사유 포함)
  사유 없는 반려              -> 차단: 사유·근거 입력 필수(반려 사유 포함)
  감사 로그 삭제              -> 차단: 감사 로그 삭제·수정 권한은 누구에게도 없음
  감사관이 승인 시도            -> 차단: 권한 없음(감사관 역할에 approve 권한 없음)
  종결된 건 재승인             -> 차단: 이미 종결된 건(승인)
  존재하지 않는 건 승인          -> 차단: 존재하지 않는 건
  상신자가 본인 건 승인(분리 원칙)   -> 차단: 상신자와 승인자 분리 원칙 위반
  반려 건을 재기안 없이 검토 진행    -> 차단: 반려된 건은 재기안 후에만 진행 가능
  타인이 대신 재기안            -> 차단: 권한 없음(부서장 역할에 redraft 권한 없음)
우회·위반 시도 차단 13/13 (100%) (목표 100%: 달성)

[역할별 권한 매트릭스(O 허용 / - 불가)]
      제안 상신 정비검토 품질검토 승인 반려 재기안 감사
시스템    O  -    -    -  -  -   -  -
정비담당   -  O    -    -  -  -   O  -
정비관    -  -    O    -  -  O   -  -
품질검사관  -  -    -    O  -  O   -  -
군수참모   -  -    -    -  O  O   -  -
감사관    -  -    -    -  -  -   -  O
... (이하 생략)
```
</details>
